## What to Vary

In [ ]:
# language="english"
# language="multilingual"
# DeepPavlov/rubert-base-cased-sentence


# raw text or vw text


# default topics (whatever)
# specific number of topics


# KeyBERTInspired
# openchat

In [1]:
from topicnet.cooking_machine import Dataset

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, TextGeneration

from umap import UMAP
from hdbscan import HDBSCAN

from hdbscan.flat import HDBSCAN_flat

from sklearn.feature_extraction.text import CountVectorizer

import gensim.corpora as corpora

from gensim.models.coherencemodel import CoherenceModel

import pandas as pd

In [40]:
import nltk
from nltk.corpus import stopwords
 
nltk.download('stopwords')

print(stopwords.words('russian'))

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/alekseev_v/nltk_data...


['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему', 'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь', 'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей', 'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя', 'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под', 'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем', 'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда', 'можно', 'при', 'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над', 'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту', 'моя', 'впр

[nltk_data]   Unzipping corpora/stopwords.zip.


In [15]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [16]:
! ls $DATA_FOLDER_PATH

20NG.csv	 MKB10__internals      RTL_Wiki_person.csv
20NG__internals  postnauka.csv	       RTL_Wiki_person__internals
api.py		 postnauka__internals  ruwiki_good__internals
Brown		 postnauka_noow.csv    ruwiki_good.txt
Brown_BOW.csv	 __pycache__	       WikiRef-220
Brown_NOOW.csv	 Reuters	       wiki_ref220_bow.csv
hf		 Reuters_BOW.csv       wiki_ref220_natural_order.csv
__init__.py	 Reuters_NOOW.csv
MKB10.csv	 RTL_Wiki.csv


In [18]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/postnauka_noow.csv',
)

dataset.get_possible_modalities()

set()

In [19]:
dataset.get_possible_modalities()

set()

In [21]:
MAIN_MODALITY = '@word'

In [22]:
dataset._data.head()

,id,raw_text,vw_text
id,,,
29998.txt,29998.txt,материал отрицательный показатель преломление ...,|@word материал отрицательный показатель прело...
7770.txt,7770.txt,культурный код экономика экономист александр а...,|@word культурный код экономика экономист алек...
32230.txt,32230.txt,faq наука третий класс факт эксперимент резуль...,|@word faq наука третий класс факт эксперимент...
27293.txt,27293.txt,обрушение волна поверхность жидкость математик...,|@word обрушение волна поверхность жидкость ма...
481.txt,481.txt,существовать ли суперсимметрия мир элементарны...,|@word существовать ли суперсимметрия мир элем...


In [23]:
dataset._data.shape

(3446, 3)

In [24]:
dataset._data.dropna(axis=0, inplace=True)

In [25]:
dataset._data.shape

(3446, 3)

In [26]:
dataset._data['raw_text']

id
29998.txt    материал отрицательный показатель преломление ...
7770.txt     культурный код экономика экономист александр а...
32230.txt    faq наука третий класс факт эксперимент резуль...
27293.txt    обрушение волна поверхность жидкость математик...
481.txt      существовать ли суперсимметрия мир элементарны...
                                   ...                        
49461.txt    пептидный белковый нейротоксин химик виктор це...
15983.txt    радиотелескоп земля космос астрофизик анатолий...
5069.txt     вояджер история полт два исследовательский зон...
31220.txt    феномен чайлдфри общество социолог ольга исупо...
9795.txt     шаг теория принятие решение книга необходимый ...
Name: raw_text, Length: 3446, dtype: object

In [27]:
docs = list(dataset._data['raw_text'].values)

In [28]:
docs[:3]

['материал отрицательный показатель преломление физик виктор веселаго распространение свет вещество фазовый групповой скорость метаматериалы различаться фазовый групповой скорость каков физика распространение свет вещество находить применение материал отрицательный показатель преломление рассказывать доктор физикоматематический наука виктор веселаго скорость распространяться энергия вещество обычно говорить излучение распространяться вещество со скорость n раз маленький n коэффициент преломление вещество коэффициент преломление n отношение скорость свет скорость распространение излучение вещество обычно уточняться распространяться распространение энергия распространение импульс происходить различный закон энергия распространяться со скорость называться групповой скорость много скорость свет эйнштейн сформулировать самый больший скорость излучение скорость свет кмс импульс распространяться фазовый скорость сколь угодно много скорость свет скорость входить соотношение emc фазовый группов

In [29]:
NUM_TOP_WORDS = 20

In [30]:
import torch
import transformers
import os

import json
import numpy as np

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [32]:
def get_phi(topic_model):
    mtw = topic_model.c_tf_idf_.todense()
    # ptw = np.array(mtw[1:, :])
    ptw = np.array(mtw[0:, :])
    pwt = ptw.T
    vocabulary = topic_model.vectorizer_model.get_feature_names_out()

    assert pwt.shape[0] == len(vocabulary)

    # topic_names = [f'topic_{i}' for i in range(pwt.shape[1])]
    topic_names = ['background_1'] + [f'topic_{i}' for i in range(NUM_TOPICS)]
    phi = pd.DataFrame(
        index=vocabulary,
        columns=topic_names,
        data=pwt,
    )

    return phi


def get_top_words(topic_model):
    mtw = topic_model.c_tf_idf_.todense()
    ptw = np.array(mtw[0:, :])
    pwt = ptw.T

    topic_names = ['background_1'] + [f'topic_{i}' for i in range(NUM_TOPICS)]
    topic_top_words = {
        n: topic_model.get_topic(t)
        for t, n in zip([-1] + list(range(NUM_TOPICS)), topic_names)
    }

    return topic_top_words


def get_dataset(topic_model, dataset, docs):
    cleaned_docs = topic_model._preprocess_text(docs)
    vectorizer = topic_model.vectorizer_model
    tokenizer = vectorizer.build_tokenizer()
    doc_tokens = [tokenizer(doc) for doc in cleaned_docs]
    doc_texts = [
        d + f' |{MAIN_MODALITY} ' + ' '.join(t)
        for d, t in zip(dataset._data.index, doc_tokens)
    ]
    data = [[d, t] for d, t in zip(dataset._data.index, doc_texts)]

    new_dataset = pd.DataFrame(
        columns=['id', 'vw_text'],
        data=data,
    )

    return new_dataset

In [41]:
NUM_TOPICS = 20
NUM_TOP_WORDS = 20
NUM_TRAINS = 20
STOP_WORDS = stopwords.words('russian')
LANGUAGE = 'multilingual'

In [42]:
! ls ../results

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [35]:
SAVE_FOLDER = os.path.join('/data_mil/shared/CompressaAI/BERTopic', 'results', 'postnauka')

In [36]:
! mkdir -p $SAVE_FOLDER

In [37]:
SAVE_FOLDER

'/data_mil/shared/CompressaAI/BERTopic/results/postnauka'

In [45]:
for seed in range(NUM_TRAINS):
    print(seed)

    seed_save_folder = os.path.join(SAVE_FOLDER, str(seed))

    os.makedirs(seed_save_folder)

    keybert = KeyBERTInspired(top_n_words=NUM_TOP_WORDS)
    mmr = MaximalMarginalRelevance(diversity=0.3, top_n_words=NUM_TOP_WORDS)
    
    representation_model = {
        "KeyBERT": keybert,
        "MMR": mmr,
    }

    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=seed)
    vectorizer_model = CountVectorizer(stop_words=STOP_WORDS)

    topic_model = BERTopic(
        language=LANGUAGE,
        top_n_words=NUM_TOP_WORDS,

        calculate_probabilities=True,
        verbose=True,
    
        umap_model=umap_model,                    # Step 2 - Reduce dimensionality
        # hdbscan_model=hdbscan_model,            # Step 3 - Cluster reduced embeddings
        vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
        representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
    )
        
    topics, probs = topic_model.fit_transform(docs)
    orig_num_topics = len(set(topic_model.topics_))
    doc_embeddings = topic_model.umap_model.embedding_

    hdbscan_model = HDBSCAN_flat(doc_embeddings, n_clusters=NUM_TOPICS)
    
    topic_model = BERTopic(
        language=LANGUAGE,
        top_n_words=NUM_TOP_WORDS,
        calculate_probabilities=True,
        verbose=True,
    
        umap_model=umap_model,                    # Step 2 - Reduce dimensionality
        hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
        vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
        representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
    )

    topics, probs = topic_model.fit_transform(docs)
    
    new_num_topics = len(set(topic_model.topics_))
    
    assert new_num_topics < orig_num_topics
    assert new_num_topics == NUM_TOPICS + 1
    assert topic_model.c_tf_idf_.shape[0] == new_num_topics
    
    phi = get_phi(topic_model)
    top_words = get_top_words(topic_model)
    new_dataset = get_dataset(topic_model, dataset, docs)
    
    phi.to_csv(f'{seed_save_folder}/phi.csv')
    
    with open(f'{seed_save_folder}/top_words.json', 'w') as f:
        f.write(
            json.dumps(
                top_words, indent=4, ensure_ascii=False
            )
        )
    
    new_dataset.to_csv(f'{seed_save_folder}/dataset.csv')

    del topic_model, phi, new_dataset

2024-03-30 02:44:50,029 - BERTopic - Embedding - Transforming documents to embeddings.


0


Batches:   0%|          | 0/108 [00:00<?, ?it/s]

2024-03-30 02:44:59,723 - BERTopic - Embedding - Completed ✓
2024-03-30 02:44:59,723 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-30 02:45:13,027 - BERTopic - Dimensionality - Completed ✓
2024-03-30 02:45:13,028 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-30 02:45:13,689 - BERTopic - Cluster - Completed ✓
2024-03-30 02:45:13,693 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-30 02:45:23,673 - BERTopic - Representation - Completed ✓
2024-03-30 02:45:25,944 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/108 [00:00<?, ?it/s]

...

19


Batches:   0%|          | 0/108 [00:00<?, ?it/s]

2024-03-30 03:08:08,442 - BERTopic - Embedding - Completed ✓
2024-03-30 03:08:08,443 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-30 03:08:23,995 - BERTopic - Dimensionality - Completed ✓
2024-03-30 03:08:23,996 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-30 03:08:24,898 - BERTopic - Cluster - Completed ✓
2024-03-30 03:08:24,905 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-30 03:08:34,870 - BERTopic - Representation - Completed ✓
2024-03-30 03:08:37,184 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/108 [00:00<?, ?it/s]

2024-03-30 03:08:47,205 - BERTopic - Embedding - Completed ✓
2024-03-30 03:08:47,207 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-30 03:09:01,615 - BERTopic - Dimensionality - Completed ✓
2024-03-30 03:09:01,616 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-30 03:09:04,027 - BERTopic - Cluster - Completed ✓
2024-03-30 03:09:04,031 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-30 03:09:09,951 - BERTopic - Representation - Completed ✓


In [46]:
! ls /data_mil/shared/CompressaAI/BERTopic/results/postnauka/0

dataset.csv  phi.csv  top_words.json


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [51]:
! tail -n 50 /data_mil/shared/CompressaAI/BERTopic/results/postnauka/0/phi.csv

ясельный,0.0,0.0,2.8492905243287476e-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ясень,0.0,0.0,1.5157445823019854e-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ясин,0.0,0.0,0.0,0.0,0.0001813921548801612,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ясир,0.0,0.0,2.8492905243287476e-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
яск,0.0,2.5336622665018527e-05,5.121037471089046e-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ясленик,0.0,0.0,1.5157445823019854e-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ясли,0.0,4.9176803732163273e-05,3.727355327953331e-05,0.0,0.0,0.0,0.0001969808121989014,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ясмин,0.0,0.0,4.1140683325298346e-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ясно,0.0009903717809521984,0.001041389083272272,0.0009190213397776743,0.000

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
